## Gömüler

Önceki örneğimizde, uzunluğu `vocab_size` olan yüksek boyutlu kelime torbası vektörleri ile işlem yapıyorduk ve düşük boyutlu konumsal temsil vektörlerinden seyrek tek-sıcak temsile açıkça dönüştürme yapıyorduk. Bu tek-sıcak temsil, bellek açısından verimli değildir, ayrıca her kelime birbirinden bağımsız olarak ele alınır, yani tek-sıcak kodlanmış vektörler kelimeler arasındaki herhangi bir anlamsal benzerliği ifade etmez.

Bu birimde, **News AG** veri kümesini keşfetmeye devam edeceğiz. Başlamak için, veriyi yükleyelim ve önceki defterden bazı tanımlamalar alalım.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## Gömme (embedding) nedir?

**Gömme** fikri, kelimeleri daha düşük boyutlu yoğun vektörlerle temsil etmektir; bu vektörler kelimenin anlamsal anlamını bir şekilde yansıtır. Daha sonra anlamlı kelime gömmelerinin nasıl oluşturulacağını tartışacağız, ancak şimdilik gömmeleri kelime vektörünün boyutunu düşürmenin bir yolu olarak düşünelim.

Yani, gömme katmanı bir kelimeyi girdi olarak alır ve belirtilen boyutta bir çıktı vektörü üretir `embedding_size`. Bir anlamda, bu, bir `Linear` katmanına çok benzer, ancak one-hot kodlu vektör yerine bir kelime numarasını girdi olarak alabilir.

Ağımızda gömme katmanını ilk katman olarak kullanarak, çanta sözcükler modelinden **embedding bag** modeline geçebiliriz; burada önce metindeki her kelime karşılık gelen gömme vektörüne dönüştürülür ve ardından tüm bu gömmeler üzerinde `sum`, `average` veya `max` gibi bazı toplama fonksiyonları hesaplanır.

![Beş dizilim kelimesi için bir gömme sınıflandırıcısını gösteren resim.](../../../../../translated_images/tr/embedding-classifier-example.b77f021a7ee67eee.webp)

Sınıflandırıcı yapay sinir ağımız gömme katmanı ile başlayacak, sonra toplama katmanı ve bunun üzerine doğrusal sınıflandırıcı gelecek:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### Değişken dizin boyutunu ele alma

Bu mimarinin bir sonucu olarak, ağımıza minibatchler belirli bir şekilde oluşturulmalıdır. Önceki bölümde, bag-of-words kullanılırken, minibatch'teki tüm BoW tensörleri, metin dizimizin gerçek uzunluğuna bakılmaksızın eşit boyutta `vocab_size` idi. Kelime gömme yöntemine geçtiğimizde, her metin örneğinde değişken sayıda kelime olacaktı ve bu örnekleri minibatch'lerde birleştirirken padding uygulamamız gerekecekti.

Bu, aynı teknik kullanılarak, veri kaynağına `collate_fn` fonksiyonunun sağlanmasıyla yapılabilir:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Gömülü sınıflandırıcı eğitimi

Artık uygun veri yükleyiciyi tanımladığımıza göre, önceki bölümde tanımladığımız eğitim fonksiyonunu kullanarak modeli eğitebiliriz:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **Not**: Burada sadece zaman açısından 25k kayıt (bir tam epoktan az) için eğitim yapıyoruz, ancak eğitimi devam ettirebilir, birkaç epok için eğitim yapan bir fonksiyon yazabilir ve daha yüksek doğruluk elde etmek için öğrenme hızı parametresi ile deney yapabilirsiniz. Yaklaşık %90 doğruluğa ulaşabilmelisiniz.


### EmbeddingBag Katmanı ve Değişken Uzunluklu Sekans Temsili

Önceki mimaride, mini partilere sığdırmak için tüm sekansların aynı uzunlukta doldurulması gerekiyordu. Bu, değişken uzunluktaki sekansları temsil etmenin en verimli yolu değildir - başka bir yaklaşım, tüm sekansların ofsetlerini tutan **offset** vektörünü kullanmaktır; bu, tek bir büyük vektörde saklanır.

![Offset sekans temsilini gösteren resim](../../../../../translated_images/tr/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Not**: Yukarıdaki resimde bir karakter dizisi gösteriyoruz, ancak örneğimizde kelime dizileri ile çalışıyoruz. Ancak, sekansları offset vektörü ile temsil etme genel prensibi aynıdır.

Offset temsili ile çalışmak için [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) katmanını kullanıyoruz. Bu, `Embedding`'e benzer, ancak giriş olarak içerik vektörü ve offset vektörünü alır ve ayrıca `mean`, `sum` veya `max` olabilen ortalama katmanını da içerir.

İşte `EmbeddingBag` kullanan değiştirilmiş ağ:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

Verisetini eğitime hazırlamak için, ofset vektörünü hazırlayacak bir dönüşüm fonksiyonu sağlamamız gerekiyor:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Önceki tüm örneklerin aksine, ağımızın artık farklı boyutlarda olan iki parametreyi kabul ettiğini unutmayın: veri vektörü ve ofset vektörü. Benzer şekilde, veri yükleyicimiz de bize 2 yerine 3 değer sağlar: hem metin hem de ofset vektörleri özellik olarak sağlanır. Bu nedenle, bunu dikkate almak için eğitim işlevimizi biraz ayarlamamız gerekiyor:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## Anlamsal Gömüler: Word2Vec

Önceki örneğimizde, model gömme katmanı kelimeleri vektör temsiline eşlemeyi öğrendi, ancak bu temsil çok fazla anlamsal anlama sahip değildi. Benzer kelimelerin veya eşanlamlıların bazı vektör mesafesi açısından (örneğin öklidyen mesafe) birbirine yakın vektörlerle karşılık geldiği bir vektör temsilini öğrenmek güzel olurdu.

Bunu yapmak için, gömme modelimizi belirli bir şekilde büyük bir metin koleksiyonunda önceden eğitmemiz gerekir. Anlamsal gömmeleri eğitmenin ilk yollarından biri [Word2Vec](https://en.wikipedia.org/wiki/Word2vec) olarak adlandırılır. Bu, kelimelerin dağıtılmış temsilini üretmek için kullanılan iki ana mimariye dayanır:

 - **Sürekli kelime torbası** (CBoW) — bu mimaride, modeli çevresel bağlamdan bir kelimeyi tahmin etmek için eğitiriz. ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$ verildiğinde, modelin amacı $(W_{-2},W_{-1},W_1,W_2)$'den $W_0$'ı tahmin etmektir.
 - **Sürekli atlama-gramı** CBoW'un tersidir. Model, mevcut kelimeyi tahmin etmek için çevresindeki bağlam kelimeleri penceresini kullanır.

CBoW daha hızlıdır, atlama-gram ise daha yavaştır ancak nadir kelimeleri temsil etmede daha iyidir.

![Hem CBoW hem de Atlama-Gram algoritmalarının kelimeleri vektörlere dönüştürmesini gösteren resim.](../../../../../translated_images/tr/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Google News veri setinde önceden eğitilmiş word2vec gömmeleriyle denemeler yapmak için **gensim** kütüphanesini kullanabiliriz. Aşağıda 'neural' kelimesine en benzer kelimeleri buluyoruz

> **Not:** Kelime vektörlerini ilk kez oluşturduğunuzda, bunların indirilmesi biraz zaman alabilir!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


Aynı zamanda kelimeden vektör gömme (embedding) hesaplayabiliriz, sınıflandırma modeli eğitilirken kullanılmak üzere (anlaşılırlık için sadece vektörün ilk 20 bileşenini gösteriyoruz):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

Semantik gömme işleminin harika yanı, vektör kodlamasını anlamsal olarak değiştirmek için manipüle edebilmenizdir. Örneğin, vektör temsili *king* ve *woman* kelimelerine mümkün olduğunca yakın, *man* kelimesinden ise mümkün olduğunca uzak olacak bir kelime bulmasını isteyebiliriz:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

Hem CBoW hem de Skip-Grams, yalnızca yerel bağlamları dikkate aldıkları için “öngörülü” gömme yöntemleridir. Word2Vec global bağlamdan faydalanmaz. 

**FastText**, Word2Vec üzerine inşa edilir ve her kelime için ve kelime içindeki karakter n-gramları için vektör temsilleri öğrenir. Temsil değerleri her eğitim adımında tek bir vektörde ortalanır. Bu, ön eğitim sürecine çok fazla ek hesaplama eklese de, kelime gömmelerinin alt kelime bilgilerini kodlamasını sağlar. 

Başka bir yöntem olan **GloVe**, birlikte oluşma matrisinin fikrinden yararlanır ve birlikte oluşma matrisini daha ifade edici ve doğrusal olmayan kelime vektörlerine ayırmak için sinirsel yöntemler kullanır.

Gensim, çeşitli kelime gömme modellerini desteklediği için, gömmeleri FastText ve GloVe olarak değiştirerek örnek üzerinde oynayabilirsiniz.


## PyTorch'ta Önceden Eğitilmiş Gömülü Temsillerin Kullanımı

Yukarıdaki örneği, gömme katmanımızdaki matrisi Word2Vec gibi anlamsal gömülü temsillerle önceden dolduracak şekilde değiştirebiliriz. Önceden eğitilmiş gömme temsili ve metin korpusumuzun sözlüklerinin muhtemelen örtüşmeyeceğini göz önünde bulundurmamız gerekir, bu yüzden eksik kelimeler için ağırlıkları rastgele değerlerle başlatacağız:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


Şimdi modelimizi eğitelim. Modelin eğitilme süresinin, önceki örneğe kıyasla önemli ölçüde daha uzun olduğunu unutmayın çünkü gömme katmanının boyutu daha büyük ve dolayısıyla parametre sayısı çok daha fazla. Ayrıca, bu yüzden aşırı öğrenmeden kaçınmak istiyorsak modelimizi daha fazla örnek üzerinde eğitmemiz gerekebilir.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

Bizim durumumuzda, oldukça farklı kelime dağarcıklarından dolayı doğrulukta büyük bir artış görmüyoruz.
Farklı kelime dağarcıkları sorununu aşmak için aşağıdaki çözümlerden birini kullanabiliriz:
* Kelime dağarcığımız üzerinde word2vec modelini yeniden eğitmek
* Önceden eğitilmiş word2vec modelinin kelime dağarcığıyla veri setimizi yüklemek. Veri setini yüklerken kullanılacak kelime dağarcığını belirtmek mümkündür.

İkinci yaklaşım daha kolay görünüyor, özellikle PyTorch `torchtext` çerçevesinin gömülü destek içermesi nedeniyle. Örneğin, GloVe tabanlı kelime dağarcığını aşağıdaki şekilde oluşturabiliriz:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Loaded vocabulary has the following basic operations:
* `vocab.stoi` dictionary allows us to convert word into its dictionary index
* `vocab.itos` does the opposite - converts number into word
* `vocab.vectors` is the array of embedding vectors, so to get the embedding of a word `s` we need to use `vocab.vectors[vocab.stoi[s]]`

Here is the example of manipulating embeddings to demonstrate the equation **kind-man+woman = queen** (I had to tweak the coefficient a bit to make it work):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

Bu gömme kullanılarak sınıflandırıcıyı eğitmek için önce veri setimizi GloVe kelime hazinesi kullanarak kodlamamız gerekiyor:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

Yukarıda gördüğümüz gibi, tüm vektör gömme işlemleri `vocab.vectors` matrisinde saklanır. Bu, gömme katmanının ağırlıklarına bu ağırlıkları basit bir kopyalama ile yüklemeyi süper kolay hale getirir:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

Şimdi modelimizi eğitelim ve daha iyi sonuçlar alıp almadığımıza bakalım:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

Doğrulukta önemli bir artış görmememizin nedenlerinden biri, veri setimizdeki bazı kelimelerin önceden eğitilmiş GloVe kelime dağarcığında eksik olması ve dolayısıyla bu kelimelerin temel olarak göz ardı edilmesidir. Bu durumu aşmak için, kendi gömme temsillerimizi veri setimiz üzerinde eğitebiliriz. 


## Bağlamsal Gömüler

Word2Vec gibi geleneksel önceden eğitilmiş gömü temsilcilerinin temel sınırlamalarından biri kelime anlamı ayrım problemidir. Önceden eğitilmiş gömüler, kelimelerin bağlamdaki bazı anlamlarını yakalayabilse de, bir kelimenin tüm olası anlamları aynı gömü içine kodlanır. Bu durum, birçok kelimenin örneğin 'play' kelimesinin kullanıldığı bağlama bağlı olarak farklı anlamlar taşıdığı için, sonrası modellerde sorunlara yol açabilir.

Örneğin, 'play' kelimesi aşağıdaki iki cümlede oldukça farklı anlamlara sahiptir:
- Tiyatroda bir **play** izledim.
- John arkadaşlarıyla **oynamak** istiyor.

Yukarıdaki önceden eğitilmiş gömüler, 'play' kelimesinin her iki anlamını da aynı gömü içinde temsil eder. Bu sınırlamanın üstesinden gelmek için, büyük bir metin korpusunda eğitilmiş ve kelimelerin farklı bağlamlarda nasıl bir araya gelebileceğini *bilen* **dil modeline** dayalı gömüler inşa etmemiz gerekir. Bağlamsal gömüler bu eğitimin kapsamı dışındadır, ancak sonraki birimde dil modellerinden bahsederken onlara tekrar döneceğiz.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
